In [ ]:
# General libraries
import os
import re
import random
import pickle
import statistics
import collections
from collections import Counter
from itertools import combinations

# Data handling
import numpy as np
import pandas as pd

# Visualization
import seaborn as sns
from matplotlib import pyplot as plt, cm, colors, colorbar
from matplotlib_venn import venn2
from mpl_toolkits.axes_grid1 import make_axes_locatable
from adjustText import adjust_text

# Machine learning & preprocessing
from sklearn.linear_model import LinearRegression, RidgeClassifier, LogisticRegression, SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.manifold import TSNE
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, silhouette_score,
    davies_bouldin_score, calinski_harabasz_score, pairwise_distances
)

# Dimensionality reduction
import umap
import umap.umap_ as umap_module  # if you need the lower-level API

# Feature selection
from boruta import BorutaPy

# Shapelet learning
from pyts.classification import LearningShapelets
from pyts.datasets import load_gunpoint
from pyts.utils import windowed_view

# Statistical tests and models
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statannot import add_stat_annotation
#import scikit_posthocs as sp
import shap
#from xgboost import XGBRegressor
from scipy import stats
from scipy.stats import (
    ttest_ind, levene, mannwhitneyu, shapiro, mstats,
    pearsonr, kruskal, skew
)
from scipy.spatial import ConvexHull, convex_hull_plot_2d

# Progress bar
from tqdm import tqdm
from statsmodels.formula.api import ols
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore') 
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, ttest_ind, f_oneway
from statsmodels.stats.multitest import multipletests
import warnings
from scipy.stats import chi2_contingency
from collections import Counter
# ----------------------------------------------------------------------
# Example initializations (optional)
scaler = StandardScaler()
mmscaler = MinMaxScaler()
pca = PCA(n_components=2, svd_solver='full')

# Classifiers
rdg = RidgeClassifier(alpha=0.5)
mlp = MLPClassifier(random_state=1, max_iter=300, activation='relu')
lgr = LogisticRegression(random_state=1, max_iter=500)
DT = DecisionTreeClassifier(random_state=0, max_depth=10)
adb = AdaBoostClassifier(n_estimators=100, random_state=0)
gbc = GradientBoostingClassifier(n_estimators=100, random_state=1)
knn = KNeighborsClassifier(n_neighbors=3)
SGD = SGDClassifier(loss='log', random_state=1, max_iter=100, early_stopping=True,
                    learning_rate='optimal', validation_fraction=0.2)
rf = RandomForestClassifier(max_depth=10, random_state=0)
clf_svm = SVC(kernel='rbf')


In [ ]:
org_directory="/home/jupy/Data_SourceFC/T0"
ls_org_directory=os.listdir(org_directory)
directory_T0=[item for item in ls_org_directory if item !='.ipynb_checkpoints' ]
directory_T0_ob=[x for x in directory_T0 if x.startswith("F")]
directory_T0_lean=[x for x in directory_T0 if x.startswith("L")]

org_directory="/home/jupy/Data_SourceFC/T45"
ls_org_directory=os.listdir(org_directory)
directory_T45=[item for item in ls_org_directory if item != '.ipynb_checkpoints']
directory_T45_ob=[x for x in directory_T45 if x.startswith("F")]
directory_T45_lean=[x for x in directory_T45 if x.startswith("L")]

In [ ]:
def extract_connectivity_subset(band, data, roi_subset):
    """
    Extract functional connectivity only for a subset of ROIs
    
    Parameters:
    -----------
    band : int
        Frequency band index to extract
    data : list
        List of file paths containing connectivity matrices
    roi_subset : list or array
        Indices of ROIs to include (e.g., [15, 20, 32, 45])
    
    Returns:
    --------
    coh_ar : numpy array
        Array of unique connectivity values for the ROI subset
        Shape: [n_subjects, n_connections]
        where n_connections = len(roi_subset) * (len(roi_subset) - 1) / 2
    """
    n_rois = len(roi_subset)
    n_connections = n_rois * (n_rois - 1) // 2  # Number of unique connections
    
    coh_ar = np.zeros([len(data), n_connections])
    
    for i in range(0, len(data)):
        # Load full connectivity matrix for the specified band
        full_matrix = np.loadtxt(data[i])[band*88:(band+1)*88, :]
        
        # Extract submatrix for selected ROIs
        sub_matrix = full_matrix[np.ix_(roi_subset, roi_subset)]
        
        # Take lower triangle (excluding diagonal) and flatten
        unique_connections = np.tril(sub_matrix, k=-1).flatten()
        
        # Remove zeros (optional: only if you want to keep only non-zero)
        # unique_connections = unique_connections[unique_connections != 0]
        
        coh_ar[i, :] = unique_connections
    coh_ar_zscored = stats.zscore(coh_ar, axis=0) 
    return coh_ar

In [ ]:
def extract_key(filename):
    return filename[0:4]
    
def align_subjects_btw_T0_T45 (directory_list_T0, directory_list_T45):
    # Create dictionaries for all time points
    time_points = {
        'T0': {extract_key(f): f for f in directory_list_T0 if extract_key(f)},
        'T45': {extract_key(f): f for f in directory_list_T45 if extract_key(f)}}    
    # Find common keys across ALL time points
    common_keys = set(time_points['T0'])  # Start with T0 keys
    for tp in time_points:
        common_keys &= set(time_points[tp])  # Intersect with each time point
        # Extract aligned files for each time point (sorted by key)
    aligned_files = {
        tp: [time_points[tp][key] for key in sorted(common_keys)]
        for tp in time_points }
    # Find unaligned files for each time point
    unique_files = {
        tp: [f for key, f in time_points[tp].items() if key not in common_keys]
        for tp in time_points}
    
    excl0=unique_files['T0']
    aligned_directory_T0=[item for item in directory_list_T0 if item not in excl0 ]
    excl45=unique_files['T45']
    aligned_directory_T45=[item for item in directory_list_T45 if item not in excl45]
    print ('Length T0: ',len(aligned_directory_T0),'    Length T45',len(aligned_directory_T45)) 
    
    # Create dictionaries with 4-digit keys
    dict_T0 = {extract_key(f): f for f in aligned_directory_T0 if extract_key(f)}
    dict_T45 = {extract_key(f): f for f in aligned_directory_T45 if extract_key(f)}
    # Find aligned keys
    common_keys = set(dict_T0) & set(dict_T45)
    aligned_T0 = [dict_T0[key] for key in common_keys]
    aligned_T45 = [dict_T45[key] for key in common_keys]
    # Find unaligned elements
    unique_T0 = [f for key, f in dict_T0.items() if key not in common_keys]
    unique_T45 = [f for key, f in dict_T45.items() if key not in common_keys]
    # Sort both aligned lists by key (first 4 digits)
    aligned_pairs = sorted(zip(aligned_T0, aligned_T45), key=lambda x: extract_key(x[0]))
    aligned_T0_sorted, aligned_T45_sorted = zip(*aligned_pairs) if aligned_pairs else ([], [])
    # Final sorted lists (now aligned by first 4 digits)
    finaldir_T0 = list(aligned_T0_sorted) 
    finaldir_T45 = list(aligned_T45_sorted) 
    
    return finaldir_T0, finaldir_T45

In [ ]:
finaldir_T0_lean,finaldir_T45_lean=align_subjects_btw_T0_T45 (directory_T0_lean, directory_T45_lean)
finaldir_T0_ob,finaldir_T45_ob=align_subjects_btw_T0_T45 (directory_T0_ob, directory_T45_ob)

In [ ]:
def show_connectivity_pattern(band, data, roi_subset, subject_idx=0):
    """
    Display the full connectivity matrix for ROI subset,
    clearly showing which connections are zero vs non-zero
    """
    # Load matrix for specific subject
    full_matrix = np.loadtxt(data[subject_idx])[band*88:(band+1)*88, :]
    sub_matrix = full_matrix[np.ix_(roi_subset, roi_subset)]
    
    # Create a dataframe for better visualization
    df = pd.DataFrame(
        sub_matrix,
        index=[f'ROI_{r}' for r in roi_subset],
        columns=[f'ROI_{r}' for r in roi_subset]
    )
    
    print(f"=== Connectivity Matrix (Subject {subject_idx}) ===")
    print(df)
    print("\n")
    
    # Create zero/non-zero indicator matrix
    zero_pattern = (sub_matrix == 0).astype(int)
    df_zero = pd.DataFrame(
        zero_pattern,
        index=[f'ROI_{r}' for r in roi_subset],
        columns=[f'ROI_{r}' for r in roi_subset]
    )
    
    print("=== Zero Pattern (1=Zero, 0=Non-zero) ===")
    print(df_zero)
    print("\n")
    
    return sub_matrix, zero_pattern


In [ ]:
os.chdir("/home/jupy/Data_SourceFC/T45")
band=4 
roi_subset = [2, 3, 8, 22, 58]

In [ ]:
cluster_label_df = pd.read_csv('/home/jupy/Subtypes_Obesity_Clustering/Satiety_Outputs/Adj45_Clinical_Test_Outputs/R_inputs/cluster_labels_gamma.csv')

idx_cluster_0 =cluster_label_df.index[cluster_label_df['cluster'] == 0].tolist()
idx_cluster_1 = cluster_label_df.index[cluster_label_df['cluster'] == 1].tolist()

print("Cluster 0 indices:", idx_cluster_0)
print("Cluster 1 indices:", idx_cluster_1)

In [ ]:
obese_Like_LeanCluster_namelist=np.array(finaldir_T45_ob)[idx_cluster_1].tolist()
obese_Diff_LeanCluster_namelist=np.array(finaldir_T45_ob)[idx_cluster_0].tolist()
Lean_namelist=finaldir_T45_lean

In [ ]:
FC_obese_Like_LeanCluster_matrix=[]

for subject_idx in range(0, len(obese_Like_LeanCluster_namelist)):
    
    matrix, zero_pattern = show_connectivity_pattern(
        band=band, 
        data= obese_Like_LeanCluster_namelist, 
        roi_subset=roi_subset,
        subject_idx=subject_idx)
    FC_obese_Like_LeanCluster_matrix.append(matrix)
    
FC_obese_Like_LeanCluster_matrix=np.array(FC_obese_Like_LeanCluster_matrix)

In [ ]:
FC_obese_Diff_LeanCluster_matrix=[]

for subject_idx in range(0, len(obese_Diff_LeanCluster_namelist)):
    
    matrix, zero_pattern = show_connectivity_pattern(
        band=band, 
        data= obese_Diff_LeanCluster_namelist, 
        roi_subset=roi_subset,
        subject_idx=subject_idx)
    FC_obese_Diff_LeanCluster_matrix.append(matrix)
    
FC_obese_Diff_LeanCluster_matrix=np.array(FC_obese_Diff_LeanCluster_matrix)
FC_obese_Diff_LeanCluster_matrix=zscore_ignore_zero(FC_obese_Diff_LeanCluster_matrix)

In [ ]:
FC_Lean_matrix=[]

for subject_idx in range(0, len(Lean_namelist)):
    
    matrix, zero_pattern = show_connectivity_pattern(
        band=band, 
        data= Lean_namelist, 
        roi_subset=roi_subset,
        subject_idx=subject_idx)
    FC_Lean_matrix.append(matrix)
    
FC_Lean_matrix=np.array(FC_Lean_matrix)
FC_Lean_matrix=zscore_ignore_zero(FC_Lean_matrix)

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def test_group_connectivity(fc_matrices, roi_subset, alpha=0.05):
    """
    fc_matrices: array/list with shape (n_subjects, n_roi, n_roi)
                 each matrix should correspond to roi_subset order
    roi_subset: list of ROI numbers, e.g. [15, 20, 32, 45]
    """

    fc_matrices = np.asarray(fc_matrices)
    n_subjects, n_roi, _ = fc_matrices.shape

    results = []

    for i in range(n_roi):
        for j in range(i + 1, n_roi):  # upper triangle only
            values = fc_matrices[:, i, j]

            mean_fc = np.mean(values)
            median_fc = np.median(values)
            n_nonzero = np.sum(values > 0)

            # Wilcoxon signed-rank test: is median FC > 0?
            # use alternative='greater' because lagged coherence is non-negative
            stat, p = wilcoxon(values, alternative='greater')

            results.append({
                "ROI_1": roi_subset[i],
                "ROI_2": roi_subset[j],
                "mean_FC": mean_fc,
                "median_FC": median_fc,
                "n_nonzero": n_nonzero,
                "prop_nonzero": n_nonzero / n_subjects,
                "p_value": p
            })

    results_df = pd.DataFrame(results)

    # FDR correction across all ROI pairs
    results_df["p_FDR"] = multipletests(
        results_df["p_value"],
        method="fdr_bh"
    )[1]

    results_df["connected_FDR"] = results_df["p_FDR"] < alpha

    return results_df

In [ ]:
Lean_results_df = test_group_connectivity(
    fc_matrices=FC_Lean_matrix,
    roi_subset=roi_subset)

Lean_results_df

Get the **mean/Median** for **Each ROI-pair** by **Averaging** all **subjects**

In [ ]:
obese_Diff_LeanCluster_results_df = test_group_connectivity(
    fc_matrices=FC_obese_Diff_LeanCluster_matrix,
    roi_subset=roi_subset)
obese_Diff_LeanCluster_results_df

In [ ]:
obese_Like_LeanCluster_results_df = test_group_connectivity(
    fc_matrices=FC_obese_Like_LeanCluster_matrix,
    roi_subset=roi_subset)

obese_Like_LeanCluster_results_df

Compare the distribution of the **values of ROI-pairs**, where
<br>each value is a **subject-wised mean/median**

# Btw Group Compare: 
## Way 1. Group-Level Network Distribution Comparison

#### 1. What is compared

For each unique ROI pair $(i,j)$, FC values are first averaged across all subjects within each group.

Each ROI pair contributes one mean value:

$$
\bar{FC}_{ij}
=
\frac{1}{n}
\sum_{s=1}^{n}
FC_{ij}^{(s)}
$$

The distribution of these mean ROI-pair values is then compared between groups.

For one group:

$$
\left[
\bar{FC}_{12},
\bar{FC}_{13},
\bar{FC}_{14},
\ldots
\right]
$$

where list length equals the number of unique ROI pairs.

---

#### 2. Formula

Mean FC for each ROI pair:

$$
\bar{FC}_{ij}
=
\frac{1}{n}
\sum_{s=1}^{n}
FC_{ij}^{(s)}
$$

Statistical comparison:

$$
\left[
\bar{FC}_{ij}
\right]^{\text{Lean}}
\quad \text{vs} \quad
\left[
\bar{FC}_{ij}
\right]^{\text{Obese}}
$$

using Kruskal–Wallis test followed by Dunn’s post hoc test.

---

#### 3. Purpose

This approach evaluates whether the overall network connectivity distribution differs between groups.

It answers:

**Does the global functional connectivity structure differ between groups?**

This is a network-level analysis and provides a global summary, but does not preserve subject-level variance.

In [ ]:

from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sp


def compare_3_groups_kw_dunn(group1, group2, group3,
                             group_names=("Group1", "Group2", "Group3")):

    g1 = np.array(group1, dtype=float)
    g2 = np.array(group2, dtype=float)
    g3 = np.array(group3, dtype=float)

    # -------------------------
    # Kruskal-Wallis
    # -------------------------
    kw_stat, kw_p = kruskal(g1, g2, g3)

    # -------------------------
    # Prepare dataframe for Dunn
    # -------------------------
    df = pd.DataFrame({
        "Value": np.concatenate([g1, g2, g3]),
        "Group": (
            [group_names[0]] * len(g1) +
            [group_names[1]] * len(g2) +
            [group_names[2]] * len(g3)
        )
    })

    # -------------------------
    # Dunn raw p-values
    # -------------------------
    dunn_raw = sp.posthoc_dunn(
        df,
        val_col="Value",
        group_col="Group",
        p_adjust=None
    )

    # -------------------------
    # Dunn FDR corrected
    # -------------------------
    dunn_fdr = sp.posthoc_dunn(
        df,
        val_col="Value",
        group_col="Group",
        p_adjust="fdr_bh"
    )

    results = {
        "KW_statistic": kw_stat,
        "KW_p_raw": kw_p,
        "Dunn_raw_p": dunn_raw,
        "Dunn_FDR_p": dunn_fdr
    }

    return results

Mean

In [ ]:
results = compare_3_groups_kw_dunn(
    group1=obese_Like_LeanCluster_results_df['mean_FC'],
    group2=obese_Diff_LeanCluster_results_df['mean_FC'],
    group3=Lean_results_df['mean_FC'],
    group_names=("Obese_Lean_like", "Obese_Lean_diff", "Lean")
)

print("Kruskal-Wallis")
print("H =", results["KW_statistic"])
print("p =", results["KW_p_raw"])

#print("\nDunn raw p-values")
#print(results["Dunn_raw_p"])

print("\nDunn FDR-corrected p-values")
print(results["Dunn_FDR_p"])
print(' ')

group_names=("Obese_Lean_like", "Obese_Lean_diff",  "Lean")
print("Group means:")
print(f"{group_names[0]}: {np.mean(obese_Like_LeanCluster_results_df['mean_FC']):.4f}")
print(f"{group_names[1]}: {np.mean(obese_Diff_LeanCluster_results_df['mean_FC']):.4f}")
print(f"{group_names[2]}: {np.mean(Lean_results_df['mean_FC']):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# -------------------------
# Prepare dataframe for plotting
# -------------------------
plot_df = pd.DataFrame({
    "Value": np.concatenate([
        obese_Like_LeanCluster_results_df['mean_FC'],
        obese_Diff_LeanCluster_results_df['mean_FC'],
        Lean_results_df['mean_FC']
    ]),
    "Group": (
        ["Obese_Lean_like"] * len(obese_Like_LeanCluster_results_df) +
        ["Obese_Lean_diff"] * len(obese_Diff_LeanCluster_results_df) +
        ["Lean"] * len(Lean_results_df)
    )
})

# Set group order
group_order = ["Obese_Lean_like", "Obese_Lean_diff", "Lean"]

# -------------------------
# Dunn FDR results
# -------------------------
dunn_fdr = results["Dunn_FDR_p"]

# -------------------------
# Custom colors
# -------------------------
box_palette = {
    "Obese_Lean_like": "lightgray",
    "Obese_Lean_diff": "lightgray",
    "Lean": "lightblue"
}

point_palette = {
    "Obese_Lean_like": "black",
    "Obese_Lean_diff": "black",
    "Lean": "blue"
}

# -------------------------
# Plot
# -------------------------
plt.figure(figsize=(8,6))

ax = sns.boxplot(
    data=plot_df,
    x="Group",
    y="Value",
    order=group_order,
    palette=box_palette,
    showfliers=False
)

sns.stripplot(
    data=plot_df,
    x="Group",
    y="Value",
    order=group_order,
    hue="Group",
    palette=point_palette,
    jitter=True,
    size=7,
    alpha=0.75,
    dodge=False
)

# remove duplicate legend
if ax.legend_ is not None:
    ax.legend_.remove()

# -------------------------
# Function to add significance bars
# -------------------------
def add_sig_bar(x1, x2, y, h, text):
    plt.plot(
        [x1, x1, x2, x2],
        [y, y+h, y+h, y],
        lw=1.5,
        c="black"
    )
    plt.text(
        (x1+x2)/2,
        y+h,
        text,
        ha='center',
        va='bottom',
        fontsize=14
    )

# -------------------------
# Add bars only if FDR < 0.05
# -------------------------
y_max = plot_df["Value"].max()
step = (plot_df["Value"].max() - plot_df["Value"].min()) * 0.12

pairs = [
    ("Obese_Lean_like", "Obese_Lean_diff", 0, 1),
    ("Obese_Lean_like", "Lean", 0, 2),
    ("Obese_Lean_diff", "Lean", 1, 2)
]

current_y = y_max + step * 0.2

for g1, g2, x1, x2 in pairs:
    p = dunn_fdr.loc[g1, g2]
    if p < 0.05:
        add_sig_bar(
            x1,
            x2,
            current_y,
            h=step * 0.2,
            text="**"
        )
        current_y += step

# -------------------------
# Labels
# -------------------------
plt.ylabel("Mean FC")
plt.xlabel("")
#plt.title("Distribution of Mean ROI-Pair FC Across Groups")

plt.tight_layout()
plt.show()